# study-design-overview — batch variant validation

Single-cell invocation of the recipe's `## Batch variant` path on the same GSE192391 fixture
(`acc="GSE192391"`). Runs Step 1 (typing) + Step 3 (verdict) only, skips figures, writes
`confounding_report.csv` into a per-study subfolder, and prints exactly one summary line.
Asserts the one-line summary prints in the documented format.

In [1]:
import os
os.makedirs("outputs", exist_ok=True)
os.chdir("outputs")
FIXTURE = "/tmp/aba-recipe-pack/recipes/genomics/study-design-overview/test-study-design-overview/sample_metadata.csv"
acc = "GSE192391"   # the study accession the orchestrator passed
print("cwd:", os.getcwd(), "| acc:", acc)

cwd: /tmp/aba-recipe-pack/recipes/genomics/study-design-overview/test-study-design-overview/outputs | acc: GSE192391


## Step 1 (typing) — verbatim

In [2]:
import pandas as pd, numpy as np, re
meta = pd.read_csv(FIXTURE)
ID_NAME = re.compile(r"\b(id|ids|patient|subject|donor|sample|accession|gsm|srr|barcode|uuid)\b", re.I)
def parses_numeric(s):
    return pd.to_numeric(s, errors="coerce").notna().mean() >= 0.8
n = len(meta)
rows = []
for c in meta.columns:
    nun  = int(meta[c].nunique(dropna=True))
    miss = float(meta[c].isna().mean())
    is_num = (not ID_NAME.search(c)) and (pd.api.types.is_numeric_dtype(meta[c]) or parses_numeric(meta[c]))
    if   nun <= 1:              role = "drop:constant"
    elif nun >= n:             role = "drop:id-like"
    elif miss > 0.5:           role = "drop:sparse"
    elif is_num and nun > 10:  role = "continuous"
    elif nun > n / 2:          role = "drop:high-cardinality"
    else:                      role = "categorical"
    rows.append(dict(column=c, dtype=str(meta[c].dtype), n_unique=nun,
                     pct_missing=round(100*miss, 1), suggested=role))
summary = pd.DataFrame(rows)
factors  = summary.loc[summary.suggested.isin(["categorical", "continuous"]), "column"].tolist()
num_cols = summary.loc[summary.suggested == "continuous", "column"].tolist()
cat_cols = [c for c in factors if c not in num_cols]
design = meta[factors].copy()
design[num_cols] = design[num_cols].apply(pd.to_numeric, errors="coerce")
for c in cat_cols:
    design[c] = design[c].astype(str)
print(f"typed {len(factors)} factors across {n} samples")

typed 6 factors across 30 samples


## Step 3 (verdict, no figures) — verbatim, minus the design_balance.png save

In [3]:
from dython.nominal import associations
import patsy
from itertools import combinations

# Step 2's association matrix is needed by Step 3 for the magnitude; batch skips the
# heatmap render but still needs the numbers (compute_only=True, no figure).
res = associations(design, nominal_columns=cat_cols, nom_nom_assoc="cramer",
                   nom_num_assoc="correlation_ratio", num_num_assoc="spearman",
                   cramers_v_bias_correction=True, nan_strategy="drop_samples",
                   compute_only=True)
mag = res["corr"].astype(float).abs().fillna(0)
m = mag.copy(); np.fill_diagonal(m.values, np.nan)
pairs = (m.stack().reset_index()
           .rename(columns={"level_0": "factor_a", "level_1": "factor_b", 0: "assoc"}))
pairs = pairs[pairs.factor_a < pairs.factor_b].sort_values("assoc", ascending=False)

TECH = re.compile(r"batch|run|lane|plate|platform|date|librar|flow.?cell|chip|machine|center|site|seq", re.I)
technical = [c for c in factors if TECH.search(c)]
def role(c): return "tech" if c in technical else "bio"
assoc_of = {tuple(sorted([r.factor_a, r.factor_b])): float(r.assoc) for r in pairs.itertuples()}
def verdict(a, b):
    d = design[[a, b]].dropna()
    X = patsy.dmatrix(f"Q('{a}') + Q('{b}')", d, return_type="dataframe")
    rank, ncol = int(np.linalg.matrix_rank(X.values)), X.shape[1]
    strength = assoc_of.get(tuple(sorted([a, b])), float("nan"))
    cat_pair = (a in cat_cols) and (b in cat_cols)
    empty = int((pd.crosstab(d[a], d[b]).values == 0).sum()) if cat_pair else None
    if   rank < ncol:        flag = "RED"
    elif strength >= 0.5:    flag = "AMBER"
    else:                    flag = "GREEN"
    return dict(factor_a=a, factor_b=b, pair_type="×".join(sorted([role(a), role(b)])),
                assoc=round(strength, 2), n=len(d), rank=rank, ncol=ncol,
                empty_cells=empty, flag=flag)
report = pd.DataFrame([verdict(a, b) for a, b in combinations(factors, 2)])
report["_k"] = report.flag.map({"RED": 0, "AMBER": 1, "GREEN": 2}) * 10 \
             + (report.pair_type != "bio×tech").astype(int)
report = (report.sort_values(["_k", "assoc"], ascending=[True, False])
                .drop(columns="_k").reset_index(drop=True))
# write per-study report into a per-study subfolder
os.makedirs(acc, exist_ok=True)
report.to_csv(os.path.join(acc, "confounding_report.csv"), index=False)
print("wrote", os.path.join(acc, "confounding_report.csv"))

wrote GSE192391/confounding_report.csv


## Batch summary line — verbatim from recipe `## Batch variant`

In [4]:
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    worst = report.iloc[0]
    n_red   = int((report.flag == "RED").sum())
    n_amber = int((report.flag == "AMBER").sum())
    print(f"{acc}: {len(factors)} factors, {n} samples | RED={n_red} AMBER={n_amber} | "
          f"worst: {worst.factor_a}×{worst.factor_b} {worst.flag}")
line = buf.getvalue().strip()
print(line)

# Assert the one-line summary printed in the documented format.
import re as _re
assert _re.match(r"^GSE192391: \d+ factors, \d+ samples \| RED=\d+ AMBER=\d+ \| worst: .+ (RED|AMBER|GREEN)$", line), \
    f"summary line does not match documented format: {line!r}"
assert n_red > 0, "expected at least one RED verdict for GSE192391"
print("\nASSERTION PASSED: batch summary line matches documented format and is non-degenerate.")

GSE192391: 6 factors, 30 samples | RED=2 AMBER=1 | worst: disease state×time RED

ASSERTION PASSED: batch summary line matches documented format and is non-degenerate.
